#### Unit Testing – Bronze Transactions
This notebook validates the Bronze ingestion of the Transactions dataset.
The source data consists of multiple CSV files (one per month), which are
ingested into a single Bronze table.

The objective of these tests is to ensure that:
- All records from all source files are ingested
- No records are lost or duplicated
- Key identifiers are present
- Ingestion completeness is verified

In [0]:
-- Row count comparison between raw volume and bronze table
SELECT
  'bronze_table' AS source_type,
  COUNT(*) AS row_count
FROM coffee.bronze.transactions

UNION ALL

SELECT
  'raw_transactions' AS source_type,
  COUNT(*) AS row_count
FROM (
  SELECT *
  FROM read_files(
    '/Volumes/workspace/default/coffee_raw_volume/transactions',
    format => 'csv',
    header => true
  )
);


In [0]:
-- aggregate validations for amount columns
SELECT
  'bronze_table' AS source_type,
  SUM(CAST(original_amount AS DOUBLE))  AS total_original_amount,
  SUM(CAST(discount_applied AS DOUBLE)) AS total_discount_amount,
  SUM(CAST(final_amount AS DOUBLE))     AS total_final_amount
FROM coffee.bronze.transactions

UNION ALL

SELECT
  'raw_transactions' AS source_type,
  SUM(CAST(original_amount AS DOUBLE))  AS total_original_amount,
  SUM(CAST(discount_applied AS DOUBLE)) AS total_discount_amount,
  SUM(CAST(final_amount AS DOUBLE))     AS total_final_amount
FROM (
  SELECT *
  FROM read_files(
    '/Volumes/workspace/default/coffee_raw_volume/transactions',
    format => 'csv',
    header => true
  )
);


In [0]:
-- Null comparison raw vs bronze
SELECT
  'bronze_table' AS source_type,
  SUM(CASE WHEN store_id IS NULL THEN 1 ELSE 0 END) AS null_store_id,
  SUM(CASE WHEN payment_method_id IS NULL THEN 1 ELSE 0 END) AS null_payment_method_id,
  SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS null_user_id
FROM coffee.bronze.transactions

UNION ALL

SELECT
  'raw_transactions' AS source_type,
  SUM(CASE WHEN store_id IS NULL THEN 1 ELSE 0 END) AS null_store_id,
  SUM(CASE WHEN payment_method_id IS NULL THEN 1 ELSE 0 END) AS null_payment_method_id,
  SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS null_user_id
FROM (
  SELECT *
  FROM read_files(
    '/Volumes/workspace/default/coffee_raw_volume/transactions',
    format => 'csv',
    header => true
  )
);
